# J4 Hybrid v2 — J3 Features + PCA + Autoencoder → LightGBM

**Pipeline**: Raw → J3 Feature Eng (~105 feat) → PCA + Autoencoder → **LightGBM** (non-linéaire)

⚡ **Runtime → GPU** pour l'autoencoder

In [ ]:
# 0. Upload Data
from google.colab import files
print('📂 Upload: X_train.csv, y_train.csv, X_test.csv')
uploaded = files.upload()
DATA_DIR = '/content'
print('\n✅', list(uploaded.keys()))

In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. J3 Feature Engineering (inlined)
# ══════════════════════════════════════════════════════════════

class FeatureGenerator:
    def fit(self, X, y=None): return self
    def transform(self, X): return X

class RawFeatureGenerator(FeatureGenerator):
    def __init__(self, cols): self.cols = cols
    def transform(self, X): return X[self.cols].copy()

class RollingStatFeatureGenerator(FeatureGenerator):
    def __init__(self, cols, windows, operations=['mean', 'std']):
        self.cols, self.windows, self.operations = cols, windows, operations
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            avail = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(avail) < w: continue
            if 'mean' in self.operations: X_new[f'RET_MEAN_{w}'] = X[avail].mean(axis=1)
            if 'std' in self.operations:  X_new[f'RET_STD_{w}'] = X[avail].std(axis=1)
            if 'min' in self.operations:  X_new[f'RET_MIN_{w}'] = X[avail].min(axis=1)
            if 'max' in self.operations:  X_new[f'RET_MAX_{w}'] = X[avail].max(axis=1)
            if 'skew' in self.operations: X_new[f'RET_SKEW_{w}'] = X[avail].skew(axis=1)
            if 'kurt' in self.operations: X_new[f'RET_KURT_{w}'] = X[avail].kurt(axis=1)
        return X_new

class MomentumGenerator(FeatureGenerator):
    def __init__(self, windows=[(1,5),(1,20),(5,20)]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for short, long in self.windows:
            cs, cl = f'RET_{short}', f'RET_{long}'
            if cs in X.columns and cl in X.columns:
                X_new[f'MOM_POINT_{short}_{long}'] = X[cs] - X[cl]
            vs = [f'RET_{i}' for i in range(1, short+1) if f'RET_{i}' in X.columns]
            vl = [f'RET_{i}' for i in range(1, long+1) if f'RET_{i}' in X.columns]
            if len(vs)==short and len(vl)==long:
                X_new[f'MOM_MA_{short}_{long}'] = X[vs].mean(axis=1) - X[vl].mean(axis=1)
        return X_new

class VolatilityRatioGenerator(FeatureGenerator):
    def __init__(self, windows=[(5,20)]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for short, long in self.windows:
            vs = [f'RET_{i}' for i in range(1, short+1) if f'RET_{i}' in X.columns]
            vl = [f'RET_{i}' for i in range(1, long+1) if f'RET_{i}' in X.columns]
            if len(vs)==short and len(vl)==long:
                ss, sl = X[vs].std(axis=1), X[vl].std(axis=1)
                X_new[f'VOL_RATIO_{short}_{long}'] = ss / (sl + 1e-9)
                if 'RET_1' in X.columns:
                    X_new[f'SHARPE_PROXY_{long}'] = X['RET_1'] / (sl + 1e-9)
        return X_new

class ShortTermInteractionGenerator(FeatureGenerator):
    def __init__(self, max_lag=10): self.max_lag = max_lag
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        cumul = 0
        for i in range(1, self.max_lag+1):
            rc, vc = f'RET_{i}', f'SIGNED_VOLUME_{i}'
            if rc in X.columns and vc in X.columns:
                term = X[rc] * X[vc]
                X_new[f'RET_x_VOL_{i}'] = term
                if i <= 5: cumul += term
        X_new['CUMUL_FLOW_5'] = cumul
        if 'MEDIAN_DAILY_TURNOVER' in X.columns:
            for i in range(1, 6):
                rc, vc = f'RET_{i}', f'SIGNED_VOLUME_{i}'
                if rc in X.columns and vc in X.columns:
                    X_new[f'RET_VOL_NORM_{i}'] = (X[rc]*X[vc]) / (X['MEDIAN_DAILY_TURNOVER']+1e-9)
        return X_new

class HigherOrderStatsGenerator(FeatureGenerator):
    def __init__(self, windows=[5,10,20]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            avail = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(avail) < w: continue
            X_new[f'RET_SKEW_{w}'] = X[avail].skew(axis=1)
            X_new[f'RET_KURT_{w}'] = X[avail].kurt(axis=1)
        return X_new

class SignFlipGenerator(FeatureGenerator):
    def __init__(self, windows=[5,10,20]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            avail = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(avail) < w: continue
            signs = np.sign(X[avail].values)
            X_new[f'SIGN_FLIPS_{w}'] = np.sum(np.diff(signs, axis=1) != 0, axis=1)
            X_new[f'POS_RATIO_{w}'] = np.mean(signs > 0, axis=1)
            streak = np.ones(len(X))
            for i in range(1, signs.shape[1]):
                streak += (signs[:,i] == signs[:,0]).astype(float) * (streak == i).astype(float)
            X_new[f'STREAK_{w}'] = streak
        return X_new

class VolumeFlowGenerator(FeatureGenerator):
    def __init__(self, windows=[5,10]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            av = [f'SIGNED_VOLUME_{i}' for i in range(1, w+1) if f'SIGNED_VOLUME_{i}' in X.columns]
            ar = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(av) >= 2:
                vols = X[av]
                X_new[f'VOL_IMBALANCE_{w}'] = vols.sum(axis=1)
                X_new[f'VOL_STD_{w}'] = vols.std(axis=1)
                half = len(av) // 2
                X_new[f'VOL_ACCEL_{w}'] = vols.iloc[:,:half].sum(axis=1) - vols.iloc[:,half:].sum(axis=1)
            if len(ar)==w and len(av)==w:
                rv, vv = X[ar].values, np.abs(X[av].values) + 1e-9
                X_new[f'VWAP_PROXY_{w}'] = np.sum(rv*vv, axis=1) / np.sum(vv, axis=1)
        if 'MEDIAN_DAILY_TURNOVER' in X.columns and 'SIGNED_VOLUME_1' in X.columns:
            X_new['VOL_TURNOVER_RATIO'] = np.abs(X['SIGNED_VOLUME_1']) / (X['MEDIAN_DAILY_TURNOVER']+1e-9)
        return X_new

class CrossLagCorrelationGenerator(FeatureGenerator):
    def __init__(self, windows=[10,20]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            ar = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            av = [f'SIGNED_VOLUME_{i}' for i in range(1, w+1) if f'SIGNED_VOLUME_{i}' in X.columns]
            if len(ar)==w and len(av)==w:
                rv, vv = X[ar].values, X[av].values
                rd, vd = rv - rv.mean(axis=1, keepdims=True), vv - vv.mean(axis=1, keepdims=True)
                num = np.sum(rd*vd, axis=1)
                den = np.sqrt(np.sum(rd**2, axis=1) * np.sum(vd**2, axis=1)) + 1e-9
                X_new[f'CORR_RET_VOL_{w}'] = num / den
            if len(ar)==w and w >= 6:
                half = w // 2
                recent, older = X[ar[:half]].values, X[ar[half:2*half]].values
                rd = recent - recent.mean(axis=1, keepdims=True)
                od = older - older.mean(axis=1, keepdims=True)
                num = np.sum(rd*od, axis=1)
                den = np.sqrt(np.sum(rd**2, axis=1) * np.sum(od**2, axis=1)) + 1e-9
                X_new[f'AUTOCORR_RET_{w}'] = num / den
        return X_new

print('Feature generators ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. Autoencoder (PyTorch GPU)
# ══════════════════════════════════════════════════════════════

class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout * 0.7),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, input_dim),
        )
    def forward(self, x): return self.decoder(self.encoder(x))
    def encode(self, x): return self.encoder(x)


def train_autoencoder(X, latent_dim=16, lr=1e-3, dropout=0.3,
                      n_epochs=100, batch_size=512, patience=10, verbose=True):
    input_dim = X.shape[1]
    n = len(X); n_val = int(n * 0.1)
    idx = np.random.RandomState(42).permutation(n)
    X_tr, X_va = X[idx[n_val:]], X[idx[:n_val]]

    train_dl = DataLoader(TensorDataset(torch.FloatTensor(X_tr)), batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(TensorDataset(torch.FloatTensor(X_va)), batch_size=batch_size*2)

    model = Autoencoder(input_dim, latent_dim, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()

    best_loss, wait, best_state = float('inf'), 0, None
    for ep in range(n_epochs):
        model.train()
        t_loss = 0
        for (b,) in train_dl:
            b = b.to(device)
            loss = crit(model(b), b)
            opt.zero_grad(); loss.backward(); opt.step()
            t_loss += loss.item() * len(b)
        t_loss /= len(X_tr)

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for (b,) in val_dl:
                b = b.to(device)
                v_loss += crit(model(b), b).item() * len(b)
        v_loss /= len(X_va)

        if v_loss < best_loss:
            best_loss, wait = v_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if verbose and (ep+1) % 10 == 0:
            print(f'    Epoch {ep+1:3d}: train={t_loss:.6f} val={v_loss:.6f}{" *" if wait==0 else ""}')
        if wait >= patience:
            if verbose: print(f'    Early stop at epoch {ep+1}')
            break

    if best_state: model.load_state_dict(best_state)
    model.eval()
    if verbose: print(f'  AE: latent={latent_dim}, best_val_loss={best_loss:.6f}')
    return model


def ae_transform(model, X, batch_size=2048):
    model.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            b = torch.FloatTensor(X[i:i+batch_size]).to(device)
            parts.append(model.encode(b).cpu().numpy())
    return np.concatenate(parts)

print('Autoencoder ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. PurgedKFold
# ══════════════════════════════════════════════════════════════

class PurgedKFold:
    def __init__(self, n_splits=5, purge_pct=0.02):
        self.n_splits, self.purge_pct = n_splits, purge_pct
    def split(self, X, y=None, groups=None):
        ut = np.sort(np.unique(groups))
        nt = len(ut); ps = int(nt * self.purge_pct); fs = nt // self.n_splits
        for i in range(self.n_splits):
            vs, ve = i*fs, ((i+1)*fs if i < self.n_splits-1 else nt)
            val_t = set(ut[vs:ve])
            excl = set(ut[max(0,vs-ps):min(nt,ve+ps)])
            tr_t = set(ut) - excl
            tr_idx = np.where(np.isin(groups, list(tr_t)))[0]
            va_idx = np.where(np.isin(groups, list(val_t)))[0]
            if len(tr_idx) > 0 and len(va_idx) > 0:
                yield tr_idx, va_idx

print('PurgedKFold ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. Load Data
# ══════════════════════════════════════════════════════════════

X_train_raw = pd.read_csv(f'{DATA_DIR}/X_train.csv')
y_train_raw = pd.read_csv(f'{DATA_DIR}/y_train.csv')
X_test_raw  = pd.read_csv(f'{DATA_DIR}/X_test.csv')

train_df = X_train_raw.merge(y_train_raw, on='ROW_ID')
y = (train_df['target'] > 0).astype(int)
X_train = train_df.drop(columns=['target', 'ROW_ID'])
X_test  = X_test_raw.drop(columns=['ROW_ID'])
test_ids = X_test_raw['ROW_ID']
ts_groups = X_train['TS'].str.extract(r'(\d+)')[0].astype(int).values

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Target balance: {y.mean():.3f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. J3 Feature Engineering
# ══════════════════════════════════════════════════════════════

generators = [
    RawFeatureGenerator(
        cols=[f'RET_{i}' for i in range(1,21)]
           + [f'SIGNED_VOLUME_{i}' for i in range(1,21)]
           + ['MEDIAN_DAILY_TURNOVER']
    ),
    RollingStatFeatureGenerator(
        cols=[f'RET_{i}' for i in range(1,21)],
        windows=[5, 10, 20],
        operations=['mean', 'std', 'min', 'max']
    ),
    MomentumGenerator(windows=[(5,20), (1,5), (1,20)]),
    VolatilityRatioGenerator(windows=[(5,20)]),
    ShortTermInteractionGenerator(max_lag=10),
    HigherOrderStatsGenerator(windows=[5, 10, 20]),
    SignFlipGenerator(windows=[5, 10, 20]),
    VolumeFlowGenerator(windows=[5, 10]),
    CrossLagCorrelationGenerator(windows=[10, 20]),
]

def build_j3_features(X, y=None, gens=None, fit=True):
    X_feat = pd.DataFrame(index=X.index)
    for gen in gens:
        if fit and hasattr(gen, 'fit'): gen.fit(X, y)
        X_feat = pd.concat([X_feat, gen.transform(X)], axis=1)
    return X_feat.loc[:, ~X_feat.columns.duplicated()]

X_train_feat = build_j3_features(X_train, y, gens=generators, fit=True)
X_test_feat  = build_j3_features(X_test, gens=generators, fit=False)
print(f'Features: {X_train_feat.shape[1]}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. Imputation + Scaling
# ══════════════════════════════════════════════════════════════

imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train_feat)
X_test_imp  = imputer.transform(X_test_feat)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

input_dim = X_train_scaled.shape[1]
print(f'Scaled: {input_dim} features')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. Train Autoencoder (fixed best-guess hyperparams)
#    Train AE once to avoid retraining at every Optuna trial
# ══════════════════════════════════════════════════════════════

AE_LATENT_DIM = 24
AE_LR = 1e-3
AE_DROPOUT = 0.3
PCA_N = 30

# PCA
pca = PCA(n_components=min(PCA_N, input_dim), random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)
print(f'PCA: {pca.n_components_} components, variance={pca.explained_variance_ratio_.sum()*100:.1f}%')

# Autoencoder
ae_model = train_autoencoder(X_train_scaled, latent_dim=AE_LATENT_DIM,
                              lr=AE_LR, dropout=AE_DROPOUT, verbose=True)
X_train_ae = ae_transform(ae_model, X_train_scaled)
X_test_ae  = ae_transform(ae_model, X_test_scaled)

# Combine: J3 features (original, unscaled for LightGBM) + PCA + AE latent
# LightGBM handles NaN natively, so we can use imputed but unscaled J3 features too
X_train_combined = np.hstack([X_train_imp, X_train_pca, X_train_ae])
X_test_combined  = np.hstack([X_test_imp, X_test_pca, X_test_ae])

feature_names = (
    list(X_train_feat.columns)
    + [f'PCA_{i}' for i in range(X_train_pca.shape[1])]
    + [f'AE_{i}' for i in range(X_train_ae.shape[1])]
)

print(f'\nCombined features: {X_train_combined.shape[1]}')
print(f'  J3 original: {X_train_imp.shape[1]}')
print(f'  PCA: {X_train_pca.shape[1]}')
print(f'  AE latent: {X_train_ae.shape[1]}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. Optuna — LightGBM on combined features
# ══════════════════════════════════════════════════════════════

N_TRIALS = 30

def objective(trial):
    params = {
        'objective': 'binary',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 8, 64),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 300),
        'subsample': trial.suggest_float('subsample', 0.4, 0.8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 50.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
    }

    pkf = PurgedKFold(n_splits=3, purge_pct=0.02)
    val_scores, train_scores = [], []
    y_arr = y.values

    for tr_idx, va_idx in pkf.split(X_train_combined, groups=ts_groups):
        model = lgb.LGBMClassifier(**params, random_state=42, n_jobs=-1)
        model.fit(X_train_combined[tr_idx], y_arr[tr_idx])
        val_scores.append(accuracy_score(y_arr[va_idx], model.predict(X_train_combined[va_idx])))
        train_scores.append(accuracy_score(y_arr[tr_idx], model.predict(X_train_combined[tr_idx])))

    val_acc = np.mean(val_scores)
    trial.set_user_attr('train_acc', np.mean(train_scores))
    trial.set_user_attr('gap', np.mean(train_scores) - val_acc)
    return val_acc

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

bp = study.best_params
print(f'\n✅ Best trial: #{study.best_trial.number}')
print(f'   Val accuracy: {study.best_value:.4f}')
print(f'   Gap: {study.best_trial.user_attrs["gap"]:.4f}')
for k, v in bp.items():
    print(f'   {k}: {v}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. Final 5-fold evaluation
# ══════════════════════════════════════════════════════════════

best_p = bp.copy()
best_p.update({'objective': 'binary', 'verbosity': -1, 'boosting_type': 'gbdt'})

pkf = PurgedKFold(n_splits=5, purge_pct=0.02)
val_s, tr_s = [], []
y_arr = y.values

for tr_i, va_i in pkf.split(X_train_combined, groups=ts_groups):
    m = lgb.LGBMClassifier(**best_p, random_state=42, n_jobs=-1)
    m.fit(X_train_combined[tr_i], y_arr[tr_i])
    val_s.append(accuracy_score(y_arr[va_i], m.predict(X_train_combined[va_i])))
    tr_s.append(accuracy_score(y_arr[tr_i], m.predict(X_train_combined[tr_i])))

print(f'5-Fold Purged CV:')
print(f'  Train: {np.mean(tr_s):.4f} | Val: {np.mean(val_s):.4f} (±{np.std(val_s):.4f}) | Gap: {np.mean(tr_s)-np.mean(val_s):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. Train Final Model & Submission
# ══════════════════════════════════════════════════════════════

final_model = lgb.LGBMClassifier(**best_p, random_state=42, n_jobs=-1)
final_model.fit(X_train_combined, y)

probs = final_model.predict_proba(X_test_combined)[:, 1]

# Calibrate threshold
target_ratio = y.mean()
sorted_probs = np.sort(probs)[::-1]
n_pos = int(len(probs) * target_ratio)
threshold = sorted_probs[min(n_pos, len(sorted_probs)-1)]
preds = (probs > threshold).astype(int)

print(f'Threshold: {threshold:.6f}')
print(f'Predictions: {preds.sum()} pos / {len(preds)} total ({preds.mean()*100:.1f}%)')

submission = pd.DataFrame({'ROW_ID': test_ids, 'score': preds})
output_file = 'submission_j4_hybrid_lgbm.csv'
submission.to_csv(output_file, index=False)
print(f'\n✅ Saved: {output_file}')

from google.colab import files
files.download(output_file)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12. Feature Importance (Top 20)
# ══════════════════════════════════════════════════════════════

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 20 features:')
print(importances.head(20).to_string(index=False))

# Count by type
pca_imp = importances[importances['feature'].str.startswith('PCA_')]['importance'].sum()
ae_imp = importances[importances['feature'].str.startswith('AE_')]['importance'].sum()
j3_imp = importances[~importances['feature'].str.match(r'^(PCA_|AE_)')]['importance'].sum()
total = pca_imp + ae_imp + j3_imp

print(f'\nImportance by source:')
print(f'  J3 features: {j3_imp/total*100:.1f}%')
print(f'  PCA:          {pca_imp/total*100:.1f}%')
print(f'  Autoencoder:  {ae_imp/total*100:.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 13. Top Trials Recap
# ══════════════════════════════════════════════════════════════

trials_df = study.trials_dataframe().sort_values('value', ascending=False).head(10)
for _, row in trials_df.iterrows():
    print(f'Trial {int(row["number"]):>2}: val={row["value"]:.4f}, gap={row["user_attrs_gap"]:.4f}')